### Finding intrinsic matrix

In [4]:
import cv2
import numpy as np
import glob
import os

# --- CONFIGURATION ---
CHECKERBOARD_DIMS = (9, 6) 
SQUARE_SIZE = 0.037  # Meters
CALIB_IMG_DIR = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/module_2_camera_calibration/resources/calibration_images'

def extract_focal_length(calib_img_dir):
    """
    Processes checkerboard images to compute the camera matrix (K), 
    distortion coefficients (D), and averaged focal length in pixels (f).
    """
    # Create a 3D grid of object points (0,0,0), (1,0,0), ...
    objp = np.zeros((CHECKERBOARD_DIMS[0] * CHECKERBOARD_DIMS[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD_DIMS[0], 0:CHECKERBOARD_DIMS[1]].T.reshape(-1, 2)
    objp = objp * SQUARE_SIZE

    objpoints = [] 
    imgpoints = [] 

    # file search to catch both .jpg and .JPG
    images = glob.glob(os.path.join(calib_img_dir, '*.[jJ][pP][gG]'))
    
    if not images:
        print(f"CRITICAL ERROR: No images found in {calib_img_dir}")
        return None, None, None

    print(f"Processing {len(images)} calibration images...")

    valid_images = 0
    gray_shape = None
    
    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue
            
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_shape = gray.shape[::-1]

        # Detect checkerboard corners
        ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD_DIMS, None)

        if ret:
            valid_images += 1
            objpoints.append(objp)
            
            # Refine corner locations to sub-pixel accuracy
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), 
                                        criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))
            imgpoints.append(corners2)
        else:
            print(f"  [SKIP] Pattern not found: {os.path.basename(fname)}")

    # Compute camera matrix
    if len(objpoints) > 0:
        print(f"\nCalibrating with {valid_images} valid images...")
        ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray_shape, None, None)
        
        # Extract focal lengths from the K matrix
        fx = mtx[0, 0]
        fy = mtx[1, 1]
        f_pixels = (fx + fy) / 2.0
        
        print(f"Calibration Complete. Reprojection Error: {ret:.4f} pixels")
        print("="*40)
        print(f"Focal Length X (fx): {fx:.2f} pixels")
        print(f"Focal Length Y (fy): {fy:.2f} pixels")
        print(f"--> Averaged Focal Length (f): {f_pixels:.2f} pixels <--")
        print("="*40 + "\n")
        
        return mtx, dist, f_pixels
    else:
        print("Calibration failed. No valid images found.")
        return None, None, None

if __name__ == "__main__":
    K_matrix, Distortion, f_pixels = extract_focal_length(CALIB_IMG_DIR)
    print(K_matrix)

Processing 27 calibration images...

Calibrating with 27 valid images...
Calibration Complete. Reprojection Error: 0.5842 pixels
Focal Length X (fx): 3065.00 pixels
Focal Length Y (fy): 3067.44 pixels
--> Averaged Focal Length (f): 3066.22 pixels <--

[[3.06499853e+03 0.00000000e+00 2.01450970e+03]
 [0.00000000e+00 3.06743951e+03 1.52774758e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]


### Match pixel coordinates from image 1 and image 2

In [16]:
import cv2
import numpy as np
import os

def compute_and_save_stereo_report(img1_path, img2_path):
    """
    Computes Fundamental (F), Essential (E), and Rotation (R) matrices for an 
    uncalibrated stereo pair and saves an annotated visualization for report submission.
    """
    
    # 1. Camera Intrinsic Matrix (K) from calibration
    K = np.array([[3.06499853e+03, 0.00000000e+00, 2.01450970e+03],
                  [0.00000000e+00, 3.06743951e+03, 1.52774758e+03],
                  [0.00000000e+00, 0.00000000e+00, 1.00000000e+00]])

    # 2. Load input stereo images
    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)
    
    if img1 is None or img2 is None:
        print("Error: Files not found. Verify image paths.")
        return

    # 3. Object Selection (Region of Interest)
    # Scaled UI for manual bounding box selection
    ui_scale = 1000 / img1.shape[1]
    img1_ui = cv2.resize(img1, (1000, int(img1.shape[0] * ui_scale)))
    
    cv2.namedWindow("Select Target Object")
    roi_coords = cv2.selectROI("Select Target Object", img1_ui, False)
    cv2.destroyWindow("Select Target Object")
    for _ in range(10): cv2.waitKey(1) # macOS GUI refresh

    # Map ROI back to original resolution
    x, y, w, h = [int(v / ui_scale) for v in roi_coords]

    # 4. Feature Detection and Correspondence Matching
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), None)
    kp2, des2 = sift.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), None)

    # Filter features strictly within the selected object boundary
    roi_kp1, roi_des1 = [], []
    for i, kp in enumerate(kp1):
        if x <= kp.pt[0] <= x+w and y <= kp.pt[1] <= y+h:
            roi_kp1.append(kp)
            roi_des1.append(des1[i])
            
    if not roi_des1:
        print("Error: No valid features detected in the selected ROI.")
        return

    # K-Nearest Neighbor matching with ratio test
    matcher = cv2.BFMatcher()
    raw_matches = matcher.knnMatch(np.array(roi_des1), des2, k=2)
    filtered_matches = sorted([m for m, n in raw_matches if m.distance < 0.75 * n.distance], 
                              key=lambda x: x.distance)

    # Enforce 8 unique spatial correspondences
    final_matches, tracked_pts = [], set()
    for m in filtered_matches:
        pt2_coord = tuple(np.round(kp2[m.trainIdx].pt, 1))
        if pt2_coord not in tracked_pts:
            tracked_pts.add(pt2_coord)
            final_matches.append(m)
        if len(final_matches) == 8: break

    if len(final_matches) < 8:
        print(f"Error: Insufficient unique matches ({len(final_matches)}/8).")
        return

    # 5. Visualization Generation (Thin Annotation Style)
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    vis_canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype='uint8')
    vis_canvas[:h1, :w1] = img1
    vis_canvas[:h2, w1:] = img2

    # Drawing parameters for report clarity
    marker_radius = 10
    line_weight = 3
    
    for m in final_matches:
        p1 = tuple(np.int32(roi_kp1[m.queryIdx].pt))
        p2 = tuple(np.int32(kp2[m.trainIdx].pt))
        p2_adj = (p2[0] + w1, p2[1])
        
        cv2.circle(vis_canvas, p1, marker_radius, (0, 0, 255), -1)
        cv2.circle(vis_canvas, p2_adj, marker_radius, (0, 0, 255), -1)
        cv2.line(vis_canvas, p1, p2_adj, (0, 255, 0), line_weight)

    # 6. Output and Data Persistence
    output_dir = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/final_uncalibrated_stereo_camera'
    output_file = os.path.join(output_dir, 'stereo_report_visual.jpg')
    
    try:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        cv2.imwrite(output_file, vis_canvas)
        print(f"Annotated report image saved: {output_file}")
    except Exception as error:
        print(f"Persistence Error: {error}")

    # 7. Geometric Matrix Computation
    pts_left = np.float32([roi_kp1[m.queryIdx].pt for m in final_matches])
    pts_right = np.float32([kp2[m.trainIdx].pt for m in final_matches])
    
    # Fundamental Matrix (F) via 8-Point Algorithm
    F_matrix, _ = cv2.findFundamentalMat(pts_left, pts_right, cv2.FM_8POINT)
    
    # Essential Matrix (E) derivation: E = K.T * F * K
    E_matrix = K.T @ F_matrix @ K
    
    # Pose Recovery: Decomposition of E into Rotation (R) and Translation (t)
    _, R_matrix, t_vec, _ = cv2.recoverPose(E_matrix, pts_left, pts_right, K)

    print("\n" + "="*50)
    print("STEREO GEOMETRY COMPUTATION RESULTS")
    print("="*50)
    print(f"Fundamental Matrix (F):\n{F_matrix}\n")
    print(f"Essential Matrix (E):\n{E_matrix}\n")
    print(f"Rotation Matrix (R):\n{R_matrix}")
    print("="*50)

    # Display final verification image
    display_w = 1200
    display_img = cv2.resize(vis_canvas, (display_w, int(vis_canvas.shape[0] * (display_w/vis_canvas.shape[1]))))
    cv2.imshow("Annotated Stereo Result - Press any key", display_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    for _ in range(10): cv2.waitKey(1)

if __name__ == "__main__":
    img_a = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/final_uncalibrated_stereo_camera/res/IMG_1879.JPG'
    img_b = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/final_uncalibrated_stereo_camera/res/IMG_1880.JPG'
    compute_and_save_stereo_report(img_a, img_b)

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Annotated report image saved: /Users/bbimali1/Documents/Computer_Vision_Spring_26/final_uncalibrated_stereo_camera/stereo_report_visual.jpg

STEREO GEOMETRY COMPUTATION RESULTS
Fundamental Matrix (F):
[[ 9.26663887e-08  8.83549203e-07 -5.00957193e-04]
 [-6.26418908e-07 -1.44442929e-07  8.33921643e-04]
 [-2.16355062e-04 -1.67558151e-03  1.00000000e+00]]

Essential Matrix (E):
[[ 0.87052807  8.30686241  3.17399081]
 [-5.88940114 -1.35909026 -1.98977948]
 [-3.02419633 -0.35685181 -0.90058441]]

Rotation Matrix (R):
[[ 0.97772574  0.00122861  0.20988298]
 [ 0.02968142  0.98912386 -0.14405903]
 [-0.20777726  0.14707984  0.96705539]]


In [ ]:
import cv2
import numpy as np

def calculate_final_distance(img1_path, img2_path):
    K = np.array([[3064.99, 0, 2014.51],
                  [0, 3067.44, 1527.75],
                  [0, 0, 1]])
                  
    BASELINE_M = 1.02  # Baseline in meters (102 cm)

    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)
    
    if img1 is None or img2 is None:
        print("Error: Files not found.")
        return

    scale = 1000 / img1.shape[1]
    img1_sel = cv2.resize(img1, (1000, int(img1.shape[0] * scale)))
    
    cv2.namedWindow("Select Object")
    roi = cv2.selectROI("Select Object", img1_sel, False)
    cv2.destroyWindow("Select Object")
    for _ in range(10): cv2.waitKey(1) 

    x, y, w, h = [int(v / scale) for v in roi]

    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), None)
    kp2, des2 = sift.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), None)

    roi_kp1, roi_des1 = [], []
    for i, kp in enumerate(kp1):
        if x <= kp.pt[0] <= x+w and y <= kp.pt[1] <= y+h:
            roi_kp1.append(kp)
            roi_des1.append(des1[i])
            
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(np.array(roi_des1), des2, k=2)
    good = sorted([m for m, n in matches if m.distance < 0.75*n.distance], key=lambda val:val.distance)

    unique, seen = [], set()
    for m in good:
        pt2 = tuple(np.round(kp2[m.trainIdx].pt, 1))
        if pt2 not in seen:
            seen.add(pt2)
            unique.append(m)
        if len(unique) == 8: break

    pts1 = np.float32([roi_kp1[m.queryIdx].pt for m in unique])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in unique])

    # --- 4. Matrices & Pose Recovery ---
    F, _ = cv2.findFundamentalMat(pts1, pts2, cv2.FM_8POINT)
    E = K.T @ F @ K
    _, R, t_unit, _ = cv2.recoverPose(E, pts1, pts2, K)

    # =======================================================
    # --- DISTANCE CALCULATION (TRIANGULATION) -----------
    # =======================================================
    
    # Scale translation vector to meters
    t_scaled = t_unit * BASELINE_M

    # Build Projection Matrices (P1 is origin, P2 is translated/rotated)
    P1 = K @ np.hstack((np.eye(3), np.zeros((3, 1))))
    P2 = K @ np.hstack((R, t_scaled))

    # Triangulate points
    points_4d = cv2.triangulatePoints(P1, P2, pts1.T, pts2.T)
    
    # Convert Homogeneous coordinates back to 3D (X, Y, Z) by dividing by W
    points_3d = points_4d[:3, :] / points_4d[3, :]

    # Get the median point to robustly represent the object's center
    median_point_3d = np.median(points_3d, axis=1) 
    
    # Calculate Euclidean distance from camera origin (0,0,0) to object
    distance = np.linalg.norm(median_point_3d)

    print("\n" + "="*50)
    print("FINAL 3D TRIANGULATION OUTPUT")
    print("="*50)
    print(f"Median 3D Point Coordinates:")
    print(f"X: {median_point_3d[0]:.4f} m")
    print(f"Y: {median_point_3d[1]:.4f} m")
    print(f"Z: {median_point_3d[2]:.4f} m\n")
    print(f"*** FINAL ESTIMATED DISTANCE: {distance:.4f} meters ({distance*100:.2f} cm) ***")
    print("="*50)

if __name__ == "__main__":
    p1 = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/final_uncalibrated_stereo_camera/res/IMG_1879.JPG'
    p2 = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/final_uncalibrated_stereo_camera/res/IMG_1880.JPG'
    calculate_final_distance(p1, p2)

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!

FINAL 3D TRIANGULATION OUTPUT
Median 3D Point Coordinates:
X: -0.2347 m
Y: -0.2422 m
Z: 2.5652 m

*** FINAL ESTIMATED DISTANCE: 2.5873 meters (258.73 cm) ***
